# Código fragmentado e explicado passo a passo
### Este documento apresenta a implementação do código dividido em etapas, com explicações detalhadas de cada parte do processo de construção da tabela verdade.

### **Célula 0:**
* Esse célula começa na primeira linha de código e estabeleze um loop que termina com novas intruções após rodar a célula 11

```
while True:

    (...)


    quer_repetir = input("Gostaria de analisar uma outra porposição? (s/n)")
    if quer_verificar.lower() == "s":
            pass

    elif quer_verificar.lower() == "n":
        print("Programa encerrado. Quando quiser reiniciar, rode o código novamente")
        break

    else:
        print("Não entendi sua resposta, tente novamente")
```

### **Célula 1:** 
1. Mensagem com instruções para o usuário e pedido de input

In [1]:
print("Os conectivos disponíveis para uso são '∧', '∨', '→', '↔' e '¬'\nVocê pode copiá-los da frase acima")
proposicao = input("Insira a proposição a ser analisada:")

Os conectivos disponíveis para uso são '∧', '∨', '→', '↔' e '¬'
Você pode copiá-los da frase acima


### **Célula 2:**
1. Separação e filtragem dos caracteres 
    * armazenados na lista "separados"
2. identificação das letras de proposição usadas
    * armazenadas em set para evitar duplicatas
    * transformadas em lista para poder ser iterada (set não tem ordem determinada, então atrapalharias as próximas etapas) 

In [52]:
separados = []
letras = set()

for i in proposicao:
    if i in ['∧', '∨', '→', '↔', '(', ')', '¬']:
        separados.append(i)
    elif i.isalpha():
        separados.append(i)
        letras.add(i)

letras = sorted(list(letras))

### **Célula 3:**
1. Definição da função "gerar_combinacoes" para construir as primeiras colunas da tabela verdade, que representam todas as possíveis combinações de valores lógicos (True e False) para um conjunto de letras de proposição

In [54]:
def gerar_combinacoes(letras, atual=[], resultado=[]):
    if len(atual) == len(letras):
        resultado.append(atual.copy())
        return resultado

    gerar_combinacoes(letras, atual + [True], resultado)
    gerar_combinacoes(letras, atual + [False], resultado)

    return resultado

### **Célula 4:**
1. Armazenamento das combinações possíveis geradas a partir da função "gerar_combinações" na variável "combinações"
2. Criação do dicionário "dados"
    * será a base para a tabela verdade
3. Criação de chaves para cada letra de proposição no dicionário "dados" associadas a listas vazias
4. Formação das colunas da tabela verdade
    * Os valores booleanos assumidos por essa letra de proposição ao longo das diferentes combinações são distribuídos nas listas correspondentes a cada proposição

In [55]:
combinacoes = gerar_combinacoes(letras)

dados = {}

for letra in letras:
    dados[letra] = []
    
for linha in combinacoes:
    for i in range(len(letras)):
        dados[letras[i]].append(linha[i])

### **Célula 5:**
1. Criação da função "converte_rpn"
    * transforma a lista dos caracteres separados e na ordem original da proposição em uma nova lista na linguagem RPN (Notação Polonesa Reversa)
    * **OBS:** RPN é uma notação pós-fixada que facilita a análise da proposição pois:
        * simplifica a análise sintática
        * possibilita fazer a análise percorrendo a expressão só uma vez, pois a estrutura passa a ser linear e direta

**O que tínhamos antes:** lista com os caracteres separados individualmente em strings na ordem em que a proposição foi escrita <br>
**O que temos agora:** nova lista na linguagem RPN (Notação Polonesa Reversa), que coloca os operadores após as letras de proposição e elimina os parênteses



In [56]:
def converte_rpn(separados):
    saida = []
    pilha = []

    precedencia = {
        '¬': 3,
        '∧': 2,
        '∨': 1,
        '→': 0,
        '↔': 0
    }

    for caractere in separados:

        if caractere.isalpha():
            saida.append(caractere)

        elif caractere == '(':
            pilha.append(caractere)

        elif caractere == ')':
            while pilha and pilha[-1] != '(':
                saida.append(pilha.pop())
            pilha.pop()

        else:
            while ((pilha and pilha[-1] != '(') and (precedencia.get(pilha[-1], 0) >= precedencia.get(caractere, 0))):
                saida.append(pilha.pop())

            pilha.append(caractere)

    while pilha:
        saida.append(pilha.pop())

    return saida

### **Célula 6:**
1. Conversão da lista "separados" em uma nova lista na linguagem RPN usando a função "converte_rpn"

In [57]:
rpn = converte_rpn(separados)

### **Célula 7:**
1. Definições de funções para aplicar cada conectivo transformando-os na linguagem de python e usando as regras de equivalência da lógica formal

**Traduções diretas de conectivos para python:**
* ∧ ⇒ and
* ∨ ⇒ or
* ¬ ⇒ not

**Regras de equivalência usadas:**

* A → B ⟺ ¬A ∨ B
* A ↔ B ⟺ (A → B) ∧ (B → A)

In [58]:
def conjuncao(v1,v2):
    resultado = (v1 and v2)
    return resultado

def disjuncao(v1,v2):
    resultado = (v1 or v2)
    return resultado

def negacao(v):
    resultado = (not v)
    return resultado

def condicional(v1,v2):
    resultado = (not(v1) or v2)
    return resultado

def bicondicional(v1,v2):
    resultado = (condicional(v1,v2) and condicional(v2,v1))
    return resultado

### **Célula 8:**
1. Criação da função "avalia_rpn" <br>
<br> 
    * itera uma lista com os caracteres da expressão em RPN a partir de um conjunto específico de valores atribuídos às proposições (uma linha da tabela verdade) <br>
    <br>
    * avalia a expressão RPN a partir do método de pilha (stack): <br>
    <br>
        * abertura do loop for que itera a expressão em RPN de caractere em caractere:
            * se o caractere for uma letra:
                * busca seu valor em valores
                * adiciona esse valor na pilha
            * se o caractere for uma negação:
                * remove 1 valor da pilha
                * aplica a negação
                * coloca o resultado de volta
            * caso contrário (se for um operador lógico que opera sobre mais de uma letra de proposição):
                * Remove dois valores da pilha:
                    * v1 = primeira letra de proposição
                    * v2 = segunda letra de proposição
                * aplica o operador correspondente
                * coloca o resultado de volta

            


In [60]:

def avalia_rpn(rpn, valores):
    pilha = []

    for caractere in rpn:

        if caractere.isalpha():
            pilha.append(valores[caractere])

        elif caractere == '¬':
            v = pilha.pop()
            pilha.append(negacao(v))

        else:
            v2 = pilha.pop()
            v1 = pilha.pop()

            if caractere == '∧':
                pilha.append(conjuncao(v1,v2))

            elif caractere == '∨':
                pilha.append(disjuncao(v1,v2))

            elif caractere == '→':
                pilha.append(condicional(v1,v2))

            elif caractere == '↔':
                pilha.append(bicondicional(v1,v2))

    return pilha[0]

### **Célula 9:**
1. Calcula os valores da coluna final da tabela verdade (que avalia a proposição como um todo)
    * percorre as linhas da tabela verdade e associa cada valor a uma letra de proposição na variável "valores"
    * aplica a função "avaliar_rpn" na expressão em RPN
    * adiciona o resultado na tabela verdade 

In [63]:
dados[proposicao] = []
for linha in combinacoes:
    valores = dict(zip(letras, linha))
    resultado = avalia_rpn(rpn, valores)
    dados[proposicao].append(resultado)


### **Célula 10:**
1. Importa a biblioteca pandas
2. Organiza em forma de tabela o dicionário "dados"
3. "Printa" a tabela verdade

In [64]:
import pandas as pd

tabela = pd.DataFrame(dados)
print(tabela) 

       A      B      C  (A ∨ B) ∧ C
0   True   True   True         True
1   True   True  False        False
2   True  False   True         True
3   True  False  False        False
4  False   True   True         True
5  False   True  False        False
6  False  False   True        False
7  False  False  False        False


### **Célula 11:**
1. Verificação de tautologia e contradição se for desejo do usuário

In [68]:
while True:
    quer_verificar = input("Gostaria de verificar se é uma tautologia, contradição ou nenhum dos casos? (s/n)")

    if quer_verificar.lower() == "s":
        if tabela[proposicao].all():
            print(f"{proposicao} é uma tautologia")
        elif not tabela[proposicao].any():
            print(f"{proposicao} é uma contradição")
        else:
            print(f"{proposicao} não é uma tautologia nem uma contradição")
        break

    elif quer_verificar.lower() == "n":
        break

    else:
        print("Não entendi sua resposta, tente novamente")
        

(A ∨ B) ∧ C não é uma tautologia nem uma contradição
